# HLLSet Algebra Tutorial Series

## Introduction & Roadmap

Welcome to the **HLLSet Algebra** tutorial series. This collection of notebooks guides you through the complete framework for probabilistic set operations, token disambiguation, lattice structures, and system dynamics.

### What is HLLSet Algebra?

HLLSet Algebra extends the HyperLogLog probabilistic data structure into a complete algebraic system that supports:

- **Ring Operations**: XOR (addition), AND (multiplication) over bit vectors
- **Lattice Structure**: Partial ordering by bitwise inclusion (the W lattice)
- **Token Disambiguation**: Recovering original tokens from fingerprints
- **Temporal Evolution**: Tracking system state changes with conservation laws

### IICA Design Principles

All modules follow the **IICA** design principles:

| Principle | Description |
|-----------|-------------|
| **I**mmutability | All committed results are frozen; operations return new instances |
| **I**dempotence | Same inputs → same outputs (deterministic hashing) |
| **C**omposability | Every stage consumes and produces HLLSets — the universal connector |
| **A**lgebraic Closure | Ring algebra provides lossless compression; set operations are closed |

## Tutorial Structure

The tutorials are organized into **4 tiers** that build progressively:

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                           TIER 4: SYSTEM DYNAMICS                           │
│  Conservation laws, monitoring, steering, evolution history                 │
│  [13] Observe → [14] Act → [15] Theory                                      │
├─────────────────────────────────────────────────────────────────────────────┤
│                      TIER 3: DISAMBIGUATION & RECONSTRUCTION                │
│  Token recovery, sequence reconstruction, lattice restoration               │
│  [08] BitVector Ring → [09] De Bruijn → [10] Disambiguation                 │
│  [11] HLLSet De Bruijn → [12] Lattice Reconstruction                        │
├─────────────────────────────────────────────────────────────────────────────┤
│                         TIER 2: SYSTEM COORDINATION                         │
│  Global registries, transactions, similarity metrics                        │
│  [05] Global Registry → [06] Ring Transaction → [07] BSS                    │
├─────────────────────────────────────────────────────────────────────────────┤
│                             TIER 1: FOUNDATION                              │
│  Core data structures and operations                                        │
│  [01] HLLSet → [02] HLLTensor → [03] HLLLattice → [04] HLLSetStore          │
└─────────────────────────────────────────────────────────────────────────────┘
```

## Tier 1: Foundation

The foundation tier introduces the core data structures.

| Tutorial | Module | Topics |
|----------|--------|--------|
| [01_hllset.ipynb](01_hllset.ipynb) | `hllset.py` | HLLSet creation, cardinality, set operations, hash configuration |
| [02_hll_tensor.ipynb](02_hll_tensor.ipynb) | `hll_tensor.py` | 2D tensor view, n-gram inscription, TokenLUT, sliding window |
| [03_hll_lattice.ipynb](03_hll_lattice.ipynb) | `hll_lattice.py` | Temporal W lattice, LatticeNode, join/meet, temporal queries |
| [04_hllset_store.ipynb](04_hllset_store.ipynb) | `hllset_store.py` | Persistent storage, derivation tracking, on-the-fly reconstruction |

### Key Concepts

- **HLLSet**: Immutable probabilistic set with ~1-2% cardinality error
- **HLLTensor**: 2D view (registers × zeros) for disambiguation
- **W Lattice**: Partial order by bitwise inclusion: $A \leq B \iff R_A \land \neg R_B = 0$
- **Content Addressing**: SHA1-based deterministic naming

## Tier 2: System Coordination

The coordination tier shows how system components work together.

| Tutorial | Module | Topics |
|----------|--------|--------|
| [05_global_registry.ipynb](05_global_registry.ipynb) | `global_registry.py` | Universe sets G₁,G₂,G₃, complement, membership, snapshots |
| [06_ring_transaction.ipynb](06_ring_transaction.ipynb) | `ring_transaction.py` | IICA transactions, ingest→commit workflow, basis selection |
| [07_bss.ipynb](07_bss.ipynb) | `bss.py` | Bell State Similarity (τ,ρ), morphism testing, directed similarity |

### Key Concepts

- **Global Registry**: Accumulates all observed n-grams across system lifetime
  - $G_1 = \bigcup_t \text{unigrams}(t)$
  - $G_2 = \bigcup_t \text{bigrams}(t)$
  - $G_3 = \bigcup_t \text{trigrams}(t)$

- **Ring Transaction**: Ephemeral workspace for IICA-compliant construction
  ```
  begin → ingest(tokens) → filter_bases() → commit() → TransactionResult
  ```

- **BSS (Bell State Similarity)**: Directed similarity metric
  - $\text{BSS}_\tau(A \to B) = |A \cap B| / |B|$ (inclusion)
  - $\text{BSS}_\rho(A \to B) = |A \setminus B| / |B|$ (exclusion)

## Tier 3: Disambiguation & Reconstruction

The disambiguation tier covers token recovery and structure restoration.

| Tutorial | Module | Topics |
|----------|--------|--------|
| [08_bitvector_ring.ipynb](08_bitvector_ring.ipynb) | `bitvector_ring.py` | Boolean ring algebra, Gaussian elimination over F₂ |
| [09_debruijn.ipynb](09_debruijn.ipynb) | `debruijn.py` | Generic De Bruijn graphs, Eulerian paths, sequence reconstruction |
| [10_disambiguation.ipynb](10_disambiguation.ipynb) | `disambiguation.py` | Token recovery, triangulation, candidate scoring |
| [11_hllset_debruijn.ipynb](11_hllset_debruijn.ipynb) | `hllset_debruijn.py` | Two-level recovery: token + HLLSet evolution |
| [12_lattice_reconstruction.ipynb](12_lattice_reconstruction.ipynb) | `lattice_reconstruction.py` | Restore W lattice from ring-compressed data |

### Key Concepts

- **Boolean Ring**: $(\mathbb{Z}/2\mathbb{Z})^N$ with XOR addition, AND multiplication
- **De Bruijn Graph**: Nodes = (k-1)-mers, Edges = k-mers
  - Eulerian path visits each edge exactly once → sequence reconstruction
  
- **Triangulation**: Intersect candidates across n-gram layers for disambiguation
  ```
  unigrams ∩ bigrams ∩ trigrams → high-confidence tokens
  ```

- **Two-Level De Bruijn**:
  - **L1 (Token)**: Nodes = bigrams, Edges = trigrams → token order
  - **L3 (Evolution)**: Nodes = HLLSet states, Edges = (D,R,N) → evolution order

## Tier 4: System Dynamics

The dynamics tier is organized by **user concern** — Observe, Act, Theory:

| Tutorial | Theme | Modules | Topics |
|----------|-------|---------|--------|
| [13_observe.ipynb](13_observe.ipynb) | **Observe** | `noether.py` + `hllset_dynamics.py` | Noether Steering Law, flux Φ(t), conservation, SystemMonitor, anomaly detection |
| [14_act.ipynb](14_act.ipynb) | **Act** | `hllset_dynamics.py` + `evolution.py` | plan_transition, find_path, SystemController, EvolutionGraph (commit/branch/merge) |
| [15_theory.ipynb](15_theory.ipynb) | **Theory** | `hllset_dynamics.py` | BitVectorState, ShiftTransition, BernoulliAnalyzer, symbolic dynamics capstone |

### Key Concepts

- **State Transition**:
  $$R(t+1) = [R(t) \setminus D(t)] \cup N(t)$$
  - $D(t)$ = deleted tokens
  - $R(t)$ = retained tokens (intersection)
  - $N(t)$ = novel tokens

- **Noether Steering Law** (Theorem 4.1):
  > If $|N(t)| = |D(t)|$ for all $t$, then $\sum_i R_i(t)$ is conserved.

- **Information Flux**: $\Phi(t) = |N(t)| - |D(t)|$
  - $\Phi > 0$: System growth
  - $\Phi < 0$: System decay
  - $\Phi \approx 0$: Balanced evolution

## Tier 5: Bayesian Interpretation (Capstone)

The Bayesian tier provides a **second reading** of every operation in the framework.
Where Tier 4 tracks *what changed* (flux, conservation), Tier 5 asks *what it means*
(probability, surprise, divergence).

| Tutorial | Theme | Modules | Topics |
|----------|-------|---------|--------|
| [16_bayesian.ipynb](16_bayesian.ipynb) | **Dual Interpretation** | `bayesian.py` | Prior/posterior, Bayes' theorem on HLLSets, temporal updating, KL divergence, evolution vs Bayesian competition (Dilution / Concentration / Hidden Shift) |
| [17_bn_ring_triangle.ipynb](17_bn_ring_triangle.ipynb) | **Love-Hate Triangle** | `bayesian_network.py` | BSS ↔ BN ↔ Ring triangle, τ = P(A\|B) identity, graph isomorphism witness, d-separation, Markov blankets, belief propagation, mutual information, forgetful functor |
| [18_markov_hll.ipynb](18_markov_hll.ipynb) | **Markov Constructs** | `markov_hll.py` | Markov chain on BSS τ-matrix, stationary distribution, PageRank, hitting times, communicating classes, HMM (forward, Viterbi), MRF (MI potentials, cliques, Gibbs energy), causal do-calculus, temporal lattice→MC |

### Key Insights

The same state transition $R(t{+}1) = [R(t) \setminus D(t)] \cup N(t)$ admits two readings:
- **Evolution** (Tier 4): absolute measures — flux $\Phi$, popcount, conservation
- **Bayesian** (Tier 5): relative measures — $P(A|U)$, surprise $S$, $D_{KL}$

The same HLLSet ring admits **three representations**:
- **Ring**: the algebraic ground truth (XOR, AND, OR, bridge law)
- **BSS Lattice**: the structural view (⊆, Hasse, levels, morphisms)
- **Bayesian Network**: the epistemic view (P, ⊥, Markov blankets, belief propagation)

These are **graph-isomorphic** ($\tau = P(A|B)$) but carry **different extra structure**.

The BSS τ-matrix is simultaneously a **Markov transition kernel** — enabling an entire
zoo of Bayesian constructs (MC, HMM, MRF, PageRank, causal models) from one formula.

## Conceptual Dependency Graph

```
Tier 1: Foundation
┌─────────┐     ┌──────────┐     ┌────────────┐     ┌──────────────┐
│ HLLSet  │───▶│ HLLTensor│───▶│ HLLLattice │───▶│ HLLSetStore  │
└─────────┘     └──────────┘     └────────────┘     └──────────────┘
     │              │                │                  │
     └──────────────┼────────────────┼──────────────────┘
                    ▼                ▼
Tier 2: Coordination
┌────────────────────┐     ┌──────────────────┐     ┌─────┐
│ GlobalNGramRegistry│───▶│ RingTransaction  │◀───│ BSS │
└────────────────────┘     └──────────────────┘     └─────┘
                                   │
                    ┌──────────────┴──────────────┐
                    ▼                             ▼
Tier 3: Disambiguation
┌──────────────┐     ┌──────────┐     ┌─────────────────────┐
│ BitVectorRing│───▶│ DeBruijn │───▶│ DisambiguationEngine│
└──────────────┘     └──────────┘     └─────────────────────┘
        │                 │                    │
        └─────────────────┼────────────────────┘
                          ▼
              ┌───────────────────────┐     ┌──────────────────────┐
              │ HLLSetDeBruijn        │───▶│ LatticeReconstruction│
              │ (token + evolution)   │     └──────────────────────┘
              └───────────────────────┘
                          │
                          ▼
Tier 4: System Dynamics (Observe / Act / Theory)
┌─────────────────────────┐   ┌──────────────────────────┐  ┌─────────────────────┐
│ 13 Observe              │   │ 14 Act                   │  │ 15 Theory           │
│ NoetherEvolution        │─▶│ plan_transition          │  │ BitVectorState      │
│ SystemMonitor           │   │ SystemController         │  │ ShiftTransition     │
│ conservation + anomaly  │   │ EvolutionGraph (commits) │  │ BernoulliAnalyzer   │
└─────────┬───────────────┘   └──────────┬───────────────┘  └──────────┬──────────┘
          │                              │                             │
          └──────────────────────────────┼─────────────────────────────┘
                                         ▼
Tier 5: Bayesian Interpretation (Capstone)
┌──────────────────────────────────────────────────────────────────────┐
│ 16 Bayesian                                                          │
│ prior / conditional / bayes_theorem  ◀── HLLSet (T01)               │
│ temporal_posterior / trajectory       ◀── HLLLattice (T03)           │
│ edge_probability / path_likelihood   ◀── DeBruijn (T08)             │
│ interpretation_divergence            ◀── NoetherEvolution (T13-15)  │
│ surprise / entropy / KL divergence   ◀── Information theory          │
├──────────────────────────────────────────────────────────────────────┤
│ 17 Love-Hate Triangle                                                │
│                                                                      │
│    ┌────────────┐                                                    │
│    │ HLLSet Ring│ (ground truth: XOR, AND, OR, bridge law)          │
│    └──────┬─────┘                                                    │
│           │                                                          │
│     ┌─────┴─────┐                                                    │
│     ▼           ▼                                                    │
│ ┌────────┐  ┌──────┐     τ(A→B) = P(A|B) = |A∩B|/|B|              │
│ │BSS Lat.│◀▶│  BN  │     graph-isomorphic, algebraically distinct   │
│ │ ORDER  │  │MEASURE│     d-separation, Markov blankets, MI          │
│ └────────┘  └──────┘                                                 │
├──────────────────────────────────────────────────────────────────────┤
│ 18 Markov Constructs                                                 │
│                                                                      │
│  BSS Lattice ──(τ matrix)──→ Markov Chain ──(stationary π)──→ BN    │
│     ORDER                      DYNAMICS                     MEASURE  │
│                                                                      │
│  ┌──────────────┐  ┌─────┐  ┌─────┐  ┌───────────┐                 │
│  │ Markov Chain │  │ HMM │  │ MRF │  │ CausalHLL │                 │
│  │ PageRank     │  │ Fwd │  │ MI  │  │ do(X)     │                 │
│  │ hitting time │  │ Vit │  │Gibbs│  │ ACE       │                 │
│  └──────────────┘  └─────┘  └─────┘  └───────────┘                 │
└──────────────────────────────────────────────────────────────────────┘
```

## Quick Start

### Prerequisites

```bash
# Build the Cython extensions
cd /path/to/hllset_algebra
python build_ext.py
```

### Minimal Example

In [1]:
# Import core modules
import sys
sys.path.insert(0, '..')

from core.hllset import HLLSet
from core.hll_lattice import HLLLattice

# Create HLLSets from token batches
doc1 = HLLSet.from_batch(["machine", "learning", "model"])
doc2 = HLLSet.from_batch(["deep", "learning", "neural"])

# Set operations
union = doc1.union(doc2)
intersection = doc1.intersect(doc2)

print(f"Doc1 cardinality: ~{doc1.cardinality():.0f}")
print(f"Doc2 cardinality: ~{doc2.cardinality():.0f}")
print(f"Union cardinality: ~{union.cardinality():.0f}")
print(f"Intersection cardinality: ~{intersection.cardinality():.0f}")

Doc1 cardinality: ~5
Doc2 cardinality: ~4
Union cardinality: ~6
Intersection cardinality: ~2


In [2]:
# Build a temporal lattice
lattice = HLLLattice(p_bits=10)

# Append observations over time
node1 = lattice.append([doc1], timestamp=1.0)
node2 = lattice.append([doc2], timestamp=2.0)

# Query cumulative state
cumulative = lattice.cumulative()
print(f"Cumulative cardinality: ~{cumulative.cardinality():.0f}")

# What changed between t=1 and t=2?
delta = lattice.delta(1.0, 2.0)
print(f"Delta (new information): ~{delta.cardinality():.0f}")

Cumulative cardinality: ~6
Delta (new information): ~3


## Learning Paths

### Path A: Core User (Tutorials 01-04)
For users who want to use HLLSets for probabilistic set operations and storage.

### Path B: System Integrator (Tutorials 01-07)
For users building applications that coordinate multiple HLLSets with transactions.

### Path C: Disambiguation Specialist (Tutorials 01-12)
For users implementing token recovery and sequence reconstruction.

### Path D: Full System (All Tutorials 01-15)
For users building complete systems with evolution tracking and conservation monitoring.

### Path E: Dual Interpretation (01, 03, 08, 13-18)
For researchers interested in the **Bayesian–Evolution duality** — how the same
algebraic operations admit complementary (and sometimes competing) interpretations.
Tutorial 17 reveals the **love-hate triangle**: Ring, BSS Lattice, and BN as three
graph-isomorphic but algebraically distinct representations of the same structure.
Tutorial 18 unleashes the **Markov construct zoo**: MC, HMM, MRF, PageRank, and
causal do-calculus — all grounded by $\tau(A \to B) = P(A|B)$.

## Module Reference

| Module | Layer | Purpose |
|--------|-------|--------|
| `hllset.py` | Foundation | Core HLLSet operations, hashing |
| `hll_tensor.py` | Foundation | 2D tensor view, TokenLUT |
| `hll_lattice.py` | Foundation | Temporal W lattice |
| `hllset_store.py` | Foundation | Persistent storage |
| `global_registry.py` | Coordination | Universe sets G₁,G₂,G₃ |
| `ring_transaction.py` | Coordination | IICA transactions |
| `bss.py` | Coordination | Bell State Similarity |
| `bitvector_ring.py` | Disambiguation | Boolean ring algebra |
| `debruijn.py` | Disambiguation | Generic De Bruijn graphs |
| `disambiguation.py` | Disambiguation | Token recovery engine |
| `hllset_debruijn.py` | Disambiguation | Two-level reconstruction |
| `lattice_reconstruction.py` | Disambiguation | W lattice restoration |
| `noether.py` | Dynamics | Conservation laws |
| `hllset_dynamics.py` | Dynamics | Monitoring & steering |
| `evolution.py` | Dynamics | State history & branching |
| `bayesian.py` | Interpretation | Bayesian priors, posteriors, KL divergence, competition |
| `bayesian_network.py` | Interpretation | BN on HLLSets — d-separation, Markov blankets, belief propagation, isomorphism witness |
| `markov_hll.py` | Interpretation | Markov chain, HMM, MRF, PageRank, causal do-calculus on HLLSets |

## How It All Connects — The Grand Architecture

The 18 tutorials form a **composable pipeline** where each tier produces the
structures that the next tier consumes. Here is the complete data-flow map.

### Tier 1 → Tier 2: Foundation feeds Coordination

| Tier 1 produces | Tier 2 consumes |
|-----------------|-----------------|
| `HLLSet` (immutable fingerprints) | `GlobalRegistry` accumulates them into G₁,G₂,G₃ |
| `HLLTensor` (register×zeros view) | `RingTransaction` selects basis vectors from tensor |
| `HLLLattice` (temporal partial order) | `BSS` computes directed similarity (τ, ρ) between lattice nodes |

### Tier 2 → Tier 3: Coordination feeds Disambiguation

| Tier 2 produces | Tier 3 consumes |
|-----------------|-----------------|
| `BSS(A→B)` directed similarity | `build_hllset_debruijn()` uses τ/ρ thresholds for edge creation |
| `RingTransaction` committed bases | `BitVectorRing` encodes them as F₂ linear combinations |
| `GlobalRegistry` universe sets | `DisambiguationEngine` uses n-gram layers for triangulation |

### Tier 3 → Tier 4: Disambiguation feeds Dynamics

This is the critical bridge — **Tutorials 11 & 12 are the diagnostic and structural
tools that Tier 4 builds its control systems upon.**

#### Tutorial 11 (HLLSet De Bruijn) → `noether.py`

| Tutorial 11 produces | `noether.py` consumes |
|---------------------|----------------------|
| `decompose_transition(H₁,H₂)` → (D, R, N) | `NoetherEvolution.step(additions=N, deletions=D)` |
| `EvolutionPhase` classification | `SteeringPhase` (refined: adds VOLATILE) |
| Information flux Φ(t) = \|N\| − \|D\| | `FluxRecord`, `flux_statistics()`, drift monitoring |
| `recover_tokens_from_drn()` | Token-level audit of what conservation law tracks |

#### Tutorial 11 (HLLSet De Bruijn) → `hllset_dynamics.py`

| Tutorial 11 produces | `hllset_dynamics.py` consumes |
|---------------------|-------------------------------|
| `decompose_transition()` | `SystemMonitor.observe()` calls it every time step |
| `classify_transition(drn)` | `SystemObservation.phase` — real-time phase detection |
| D/R/N cardinalities | `TransitionPlan.cost` = \|D\| + \|N\| (steering effort) |
| `FullDRNTriple` with tokens | `SystemController.compute_action()` — precise token-level steering |
| `DRNTriple` → `ShiftTransition.from_drn()` | `BernoulliAnalyzer` — symbolic dynamics view of evolution |

#### Tutorial 12 (Lattice Reconstruction) → `evolution.py`

| Tutorial 12 produces | `evolution.py` consumes |
|---------------------|------------------------|
| `BSSMorphismGraph` (pairwise BSS edges) | `EvolutionGraph` — same DAG pattern with content-addressed nodes |
| `ReconstructedLattice` (partial order) | `commit()` / `rollback()` / `checkout()` navigate the partial order |
| `lattice_to_dot()` visualization | `EvolutionGraph.to_dot()` — same DOT export pattern |
| Hasse diagram (transitive reduction) | `find_common_ancestor()` walks the reduced edge set |
| Level structure (antichains) | Branch tips are antichains in the evolution DAG |

#### Tutorial 12 (Lattice Reconstruction) → `hllset_dynamics.py`

| Tutorial 12 produces | `hllset_dynamics.py` consumes |
|---------------------|-------------------------------|
| Waypoint HLLSets from lattice levels | `find_path(source, target, waypoints)` — Dijkstra over lattice |
| BSS-based reachability | `TransitionPlan.is_reachable` — τ > 0 check |
| Subset detection (A ⊆ B) | `SystemController` convergence — τ ≥ threshold means "at target" |

### Tier 4 → Tier 5: Dynamics feeds Interpretation

| Tier 4 produces | Tier 5 (Bayesian) consumes |
|-----------------|---------------------------|
| `NoetherEvolution` states + flux | `interpretation_divergence()` — compares evolution vs Bayesian verdicts |
| `SteeringPhase` (GROWTH/DECAY/BALANCED) | Bayesian "competition" — same transition, different conclusions |
| `EvolutionGraph` branches | Bayesian branches = competing hypotheses with prior probabilities |
| `HLLLattice` cumulative states | `temporal_posterior()` — $P_t(A)$ tracks probability over time |
| De Bruijn edge multiplicities | `edge_probability()`, `path_log_likelihood()` — Bayesian path selection |

### Tier 5 Internal: Bayesian → BN Triangle → Markov Constructs (Tutorials 16 → 17 → 18)

| Tutorial 16 produces | Tutorial 17 consumes |
|---------------------|---------------------|
| `conditional(A, B)` = $P(A\|B)$ | `HLLBayesNet.from_hllsets()` — structure learning via pairwise conditionals |
| `prior(A, U)` = $P(A)$ | `belief_propagation()` — prior beliefs before evidence |
| `bayes_theorem()` | Markov blanket reasoning — Bayes' update over neighborhoods |
| `surprise()` / `kl_divergence()` | `mutual_information()` — information-theoretic distance |

| Tutorial 17 produces | Tutorial 18 consumes |
|---------------------|---------------------|
| `HLLBayesNet` (DAG structure) | `CausalHLL(bn).do('X')` — causal intervention via graph surgery |
| `hllset_mutual_information()` | `MarkovRandomField` — MI-based edge potentials |
| `belief_propagation()` | `CausalHLL.do()` — belief prop in mutilated graph |
| τ(A→B) = P(A\|B) identity | `HLLMarkovChain` — transition kernel T[i,j] = τ(Sⱼ→Sᵢ) |

And **BSS** (Tier 2) feeds **BN** (Tier 5) via the isomorphism:

| BSS produces | BN / Markov consumes |
|-------------|-------------|
| `BSSMorphismGraph` | `HLLBayesNet.from_bss_graph()` — the isomorphism constructor |
| τ(A→B) weights | CPT entries: P(A\|B) = τ(A→B) |
| `bss_matrix()` | `HLLMarkovChain.from_bss_matrix()` — direct MC construction |
| `morphism_graph()` | `isomorphism_witness()` — formal graph equivalence proof |

### The Complete Pipeline

```
  Tier 1                    Tier 2                   Tier 3                     Tier 4                    Tier 5
┌──────────┐          ┌──────────────┐          ┌──────────────────┐       ┌─────────────────┐       ┌─────────────┐
│ HLLSet   │──────▶  │ BSS (τ,ρ)    │──────▶  │ HLLSet DeBruijn  │─────▶│ Noether         │─────▶│ Bayesian    │
│ HLLTensor│──────▶  │ Registry     │──────▶  │  D/R/N triples   │─────▶│  Φ(t) flux      │       │  P(A|U)     │
│ Lattice  │──────▶  │ Transaction  │──────▶  │  Evolution paths │─────▶│  Conservation   │       │  D_KL       │
│ Store    │          └──────┬───────┘          │                  │       │  Steering Law   │       │  Surprise   │
└──────────┘                 │                  │ Lattice Recon    │─────▶│                 │       │  Divergence │
   [01-04]                   │                  │  Partial order   │       │ Dynamics        │       │  Temporal   │
                             │                  │  Hasse diagram   │─────▶│  Monitor/Steer  │─────▶│  posteriors │
                             │                  │  Level structure │       │  Plan paths     │       │             │
                             │                  └──────────────────┘       │                 │       │ Competition │
                             │                     [08-12]                 │ Evolution       │─────▶│  Dilution   │
                             │                                             │  Commits/Branch │       │  Concentr.  │
                             │                                             │  Merge/Rollback │       │  Hidden     │
                             │                                             └─────────────────┘       │  Shift      │
                             │                                                [13-15]                ├─────────────┤
                             │                                                                       │ BN Triangle │
                             └───────────────────────────────────────────────────────────────────────▶│ τ = P(A|B) │
                              BSS → BN isomorphism                                                   │ d-separation│
                              BSS → MC transition kernel                                             │ Markov blnkt│
                                                                                                     │ Belief prop │
                                                                                                     ├─────────────┤
                                                                                                     │ Markov Zoo  │
                                                                                                     │ MC/PageRank │
                                                                                                     │ HMM/Viterbi │
                                                                                                     │ MRF/Gibbs   │
                                                                                                     │ Causal do() │
                                                                                                     └─────────────┘
                                                                                                       [16-18]
```

### The Unifying Equation

The entire framework revolves around one state transition:

$$R(t+1) = \bigl[R(t) \setminus D(t)\bigr] \cup N(t)$$

- **Tier 1** defines what $R$, $D$, $N$ *are* (HLLSets with ring/lattice algebra)
- **Tier 2** measures *similarity* between them (BSS) and *coordinates* their construction (transactions)
- **Tier 3** *recovers* the tokens inside $D$, $R$, $N$ and *reconstructs* their ordering
- **Tier 4** *monitors* the flux $\Phi = |N| - |D|$, *enforces* conservation, and *steers* the system
- **Tier 5** *interprets* probability $P(A|U)$, measures *surprise* and *divergence*, reveals when conservation and inference *compete*, exposes the **love-hate triangle** between Ring, BSS, and BN, and unleashes the **Markov construct zoo** — MC, HMM, MRF, PageRank, and causal do-calculus — all from one identity: $\tau = P(A|B)$

### The Love-Hate Triangle (Tutorial 17) → Markov Zoo (Tutorial 18)

$$\boxed{\tau(A \to B) \;=\; P(A \mid B) \;=\; \frac{|A \cap B|}{|B|}}$$

Three representations of one ring:
- **Ring**: ground truth — $\oplus$, $\wedge$, $\vee$, bridge law, Gaussian elimination
- **BSS Lattice**: ORDER — $\subseteq$, Hasse diagram, levels, morphisms $(τ, ρ)$
- **Bayesian Network**: MEASURE — $P$, $\perp$, Markov blankets, belief propagation

Graph-isomorphic via $\tau = P(A|B)$. Algebraically distinct (BSS has ρ; BN has d-separation).

The same $\tau$-matrix serves as **Markov transition kernel**, enabling:
- **Markov Chain**: stationary π, PageRank, hitting times, communicating classes
- **HMM**: hidden states = true sets, observations = HLL estimates, Viterbi decoding
- **MRF**: edge potential = $\exp(I(S_i; S_j))$, Gibbs energy, maximal cliques
- **Causal Model**: $P(Y | \text{do}(X))$ via graph surgery on HLLBayesNet

## Getting Help

- **Documentation**: See the `DOCS/` folder for architecture and theory
- **Theory**: `DOCS/HLLSet_Ring_Lattice.md` for mathematical foundations
- **Introduction**: `DOCS/INTRODUCTION_TO_HLLSETS.md` for conceptual overview

---

**Ready to start?** Begin with [Tutorial 01: HLLSet](01_hllset.ipynb)!